In [1]:
# 12_degree_baseline.ipynb
# =============================================================================
# Round-3 closure of the connectivity argument. If the relational gain is mostly
# CONNECTIVITY rather than specific wiring, then simply giving the non-graph model
# each firm's CONNECTION COUNTS (its degree in each relation group) should recover
# most of the gain -- without any message passing. We test exactly this, and we
# firm up the degree-preserving null with 50 draws.
#
#   intra                 : 23-d company features, no graph (baseline).
#   intra + degree        : 23-d + per-relation-group connection counts, no graph.
#   intra + neighbor max  : 23-d + max-pooled neighbor embeddings, no graph (ref).
#   graph (0-11)          : the GNN (reference ceiling).
#   degree-preserving null: 50 draws (reference).
#
# If intra+degree approaches the GNN, a practitioner does not need a GNN to capture
# this signal -- the positive, practical takeaway of the paper.
# 10 seeds, mask A. Figure grayscale, dpi 600, png + pdf, no caption.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(10))
N_PERM = 50


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


def ks_stat(y, s):
    fpr, tpr, _ = roc_curve(y, s); return float(np.max(tpr - fpr))


def metrics(y, s, thr=0.5):
    return {"AUC": roc_auc_score(y, s), "F1": f1_score(y, (s >= thr).astype(int),
            zero_division=0), "KS": ks_stat(y, s)}


data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
N_COMP = comp.num_nodes
person_x = data["person"].x.numpy(); PDIM = person_x.shape[1]
Xbase = comp.x.numpy().astype(np.float64)
Xbase[:, :2] = np.log1p(np.clip(Xbase[:, :2], 0, None))
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask
ET_0_11 = [et for et in data.edge_types if int(et[1].split("_")[1]) <= 11]
IN_DIMS = {"company": comp.x.size(1), "person": PDIM}

# --- Degree features per company (connection counts by relation group) -------
FF = {6, 7, 10, 11}          # firm-to-firm economic
PM = {0, 1, 2, 3, 4, 5, 8, 9}  # person-mediated economic
deg_ff = np.zeros(N_COMP); deg_pm = np.zeros(N_COMP)
pmax = np.full((N_COMP, PDIM), -np.inf); pcnt = np.zeros(N_COMP)
for et in data.edge_types:
    s, r, t = et; rel = int(r.split("_")[1]); ei = data[et].edge_index.numpy()
    # company endpoints get +1 degree
    for side, nt in [(0, s), (1, t)]:
        if nt == "company":
            nodes = ei[side]
            if rel in FF:
                np.add.at(deg_ff, nodes, 1.0)
            elif rel in PM:
                np.add.at(deg_pm, nodes, 1.0)
    # neighbor max-pool (person neighbors) for reference config
    if s == "company" and t == "person":
        cc, pp = ei[0], ei[1]
    elif s == "person" and t == "company":
        pp, cc = ei[0], ei[1]
    else:
        continue
    for c, p in zip(cc, pp):
        pmax[c] = np.maximum(pmax[c], person_x[p]); pcnt[c] += 1
pmax[~np.isfinite(pmax)] = 0.0
deg_total = deg_ff + deg_pm
degree_feats = np.column_stack([np.log1p(deg_ff), np.log1p(deg_pm), np.log1p(deg_total),
                                (deg_total > 0).astype(float)]).astype(np.float32)
print(f"degree features: 4 (log firm-to-firm, log person-mediated, log total, has-any)")
print(f"  mean economic degree = {deg_total.mean():.2f}, max = {deg_total.max():.0f}")


class HeteroGNN(nn.Module):
    def __init__(self, edge_types, in_dims, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([HeteroConv({et: SAGEConv((hidden, hidden), hidden)
                     for et in edge_types}, aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, xd, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training) for nt, v in h.items()}
        return self.head(h["company"])


class MLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def cw():
    cls = torch.bincount(y[trm], minlength=2).float(); return cls.sum() / (2 * cls.clamp(min=1))


def run(kind, seed, feat=None, eid=None, return_prob=False):
    torch.manual_seed(seed); np.random.seed(seed); w = cw()
    if kind == "gnn":
        Xc = StandardScaler().fit(Xbase[trm.numpy()]).transform(Xbase).astype(np.float32)
        xd = {"company": torch.tensor(Xc), "person": data["person"].x}
        model = HeteroGNN(list(eid.keys()), IN_DIMS); fwd = lambda: model(xd, eid)
    else:
        sc = StandardScaler().fit(feat[trm.numpy()]); X = torch.tensor(sc.transform(feat).astype(np.float32))
        model = MLP(X.size(1)); fwd = lambda: model(X)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    bv, bs, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(fwd()[trm], y[trm], weight=w); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(fwd(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), p[vam.numpy()])
        if va > bv:
            bv, bs, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(bs); model.eval()
    with torch.no_grad():
        p = F.softmax(fwd(), 1)[:, 1].numpy()
    m = metrics(y[tem.numpy()].numpy(), p[tem.numpy()])
    return (m, p[tem.numpy()]) if return_prob else m


# --- E9: configurations ------------------------------------------------------
Xdeg = np.concatenate([Xbase, degree_feats], 1)
Xmax = np.concatenate([Xbase, pmax.astype(np.float32)], 1)
real_eid = {et: data[et].edge_index for et in ET_0_11}
configs = {
    "intra": ("mlp", Xbase, None),
    "intra+degree": ("mlp", Xdeg, None),
    "intra+neighbor_max": ("mlp", Xmax, None),
    "graph_0_11": ("gnn", None, real_eid),
}
rows = []
for seed in SEEDS:
    for name, (kind, feat, eid) in configs.items():
        rows.append({"config": name, "seed": seed, **run(kind, seed, feat=feat, eid=eid)})
raw = pd.DataFrame(rows)
raw.to_csv(TAB / "12_degree_baseline_raw.csv", index=False)
summ = raw.groupby("config").agg(AUC_mean=("AUC", "mean"), AUC_std=("AUC", "std"),
                                 F1_mean=("F1", "mean"), KS_mean=("KS", "mean")).reset_index().round(4)
summ.to_csv(TAB / "12_degree_baseline.csv", index=False)
print("\n=== E9: degree-feature baseline (mask A, 10 seeds) ===")
mm = summ.set_index("config")["AUC_mean"]
for c in ["intra", "intra+degree", "intra+neighbor_max", "graph_0_11"]:
    print(f"  {c:20s} AUC {summ.set_index('config').loc[c,'AUC_mean']:.3f} "
          f"+/- {summ.set_index('config').loc[c,'AUC_std']:.3f}")
intra_a, deg_a, graph_a = mm["intra"], mm["intra+degree"], mm["graph_0_11"]
recovered = (deg_a - intra_a) / (graph_a - intra_a) * 100
print(f"  degree features recover {recovered:.0f}% of the intra->graph gain "
      f"({intra_a:.3f} -> {deg_a:.3f} vs graph {graph_a:.3f})")
# paired t: does the GNN still beat intra+degree?
p_intra = raw[raw.config == "intra"].sort_values("seed")["AUC"].values
p_deg = raw[raw.config == "intra+degree"].sort_values("seed")["AUC"].values
p_graph = raw[raw.config == "graph_0_11"].sort_values("seed")["AUC"].values
t1, pp1 = ttest_rel(p_deg, p_intra)
t2, pp2 = ttest_rel(p_graph, p_deg)
print(f"  intra+degree vs intra: +{(p_deg-p_intra).mean():.3f} (p={pp1:.2e})")
print(f"  graph vs intra+degree: +{(p_graph-p_deg).mean():.3f} (p={pp2:.3f}) "
      f"-> {'GNN adds little beyond degree' if pp2>0.05 else 'GNN still adds a little'}")

# --- E10: 50-draw degree-preserving null -------------------------------------
def dpn_eid(seed):
    g = np.random.default_rng(5000 + seed); ed = {}
    for et in ET_0_11:
        ei = data[et].edge_index.numpy().copy(); ei[1] = g.permutation(ei[1])
        ed[et] = torch.tensor(ei, dtype=torch.long)
    return ed


dpn = np.array([run("gnn", s % 10, eid=dpn_eid(s))["AUC"] for s in range(N_PERM)])
pd.DataFrame({"draw": range(N_PERM), "AUC": dpn.round(4)}).to_csv(TAB / "12_degree_null_50.csv", index=False)
emp_p = float((dpn >= graph_a).mean())
print(f"\n=== E10: 50-draw degree-preserving null ===")
print(f"  null AUC {dpn.mean():.3f} +/- {dpn.std():.3f} | graph {graph_a:.3f} | "
      f"empirical p(null>=graph) = {emp_p:.3f} (n={N_PERM})")

# --- Figure ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.5))
order = ["intra", "intra+degree", "intra+neighbor_max", "graph_0_11"]
labels = ["Intra", "Intra +\ndegree", "Intra +\nneighbor max", "GNN\n(graph)"]
means = [mm[c] for c in order]; stds = [summ.set_index("config").loc[c, "AUC_std"] for c in order]
ax.bar(range(4), means, yerr=stds, color=["#999999", "#555555", "#666666", "#222222"],
       edgecolor="black", linewidth=0.5, error_kw={"elinewidth": 0.9, "capsize": 3}, width=0.62)
ax.axhline(intra_a, color="black", linestyle=":", linewidth=0.8)
ax.set_xticks(range(4)); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Test AUC (mask A)"); ax.set_ylim(0.6, 0.8)
savefig(fig, "fig_degree_baseline")

print("\n12_degree_baseline complete: degree-feature baseline + 50-draw null; 1 figure, 3 tables.")


degree features: 4 (log firm-to-firm, log person-mediated, log total, has-any)
  mean economic degree = 3.98, max = 46



=== E9: degree-feature baseline (mask A, 10 seeds) ===
  intra                AUC 0.686 +/- 0.008
  intra+degree         AUC 0.809 +/- 0.006
  intra+neighbor_max   AUC 0.748 +/- 0.015
  graph_0_11           AUC 0.767 +/- 0.014
  degree features recover 152% of the intra->graph gain (0.686 -> 0.809 vs graph 0.767)
  intra+degree vs intra: +0.123 (p=4.53e-11)
  graph vs intra+degree: +-0.042 (p=0.000) -> GNN still adds a little



=== E10: 50-draw degree-preserving null ===
  null AUC 0.752 +/- 0.012 | graph 0.767 | empirical p(null>=graph) = 0.120 (n=50)



12_degree_baseline complete: degree-feature baseline + 50-draw null; 1 figure, 3 tables.
